# zipcast on Kaggle: epub → audiobook (Qwen3-TTS)

This notebook runs the same zipcast web GUI on a Kaggle GPU notebook. Upload epubs, pick chapters, tune chunk/batch size, preview voices, and watch live progress from the browser GUI.

**Before running:**
1. In Kaggle, turn **Accelerator** on and choose a GPU.
2. Turn **Internet** on in the notebook settings, otherwise cloning/installing/tunneling will fail.
3. Run every cell top to bottom.
4. The launch cell prints a temporary `https://*.trycloudflare.com` link. Open it, upload `.epub` files, and start conversion.

Generated `.m4b` files are written under `/kaggle/working`, so they are visible in the notebook output files panel too.


In [ ]:
import os
import sys

# reduce CUDA memory fragmentation on long multi-chapter runs (must be set
# before torch initializes a CUDA context)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/VinishReddyK/zipcast-epub.git"  # @param {type:"string"}
REPO_DIR = "/kaggle/working/zipcast"
CONTENT_DIR = "/kaggle/working/zipcast_content"

os.environ["ZIPCAST_CONTENT_DIR"] = CONTENT_DIR
os.environ["ZIPCAST_WORK_DIR"] = f"{CONTENT_DIR}/zipcast_work"
os.environ["ZIPCAST_VOICES_DIR"] = f"{CONTENT_DIR}/zipcast_voices"

os.makedirs(CONTENT_DIR, exist_ok=True)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# install ffmpeg (for m4b muxing) and the epub-parsing + Qwen3-TTS + web GUI dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg wget
!pip install -q -r {REPO_DIR}/colab/requirements.txt


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. In Kaggle's notebook settings, enable a GPU accelerator; CPU synthesis will be very slow.")


## Optional: preload the named-speaker model

Run this cell if you want to pay the model load cost now. You can also skip it; the model will load when the first conversion starts, and the GUI will show those logs.


In [ ]:
from colab import webapp

PRELOAD_MODEL = False  # @param {type:"boolean"}
MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"  # @param {type:"string"}

if PRELOAD_MODEL:
    webapp.preload(model_name=MODEL_NAME, device=device, on_progress=print)
    print("\nmodel preloaded -- conversions will start without waiting on a cold load.")
else:
    print("skipping preload; model load logs will appear in the GUI when conversion starts.")


## Launch the GUI

This starts the local Flask web app and opens a free Cloudflare quick tunnel to it. Keep this cell running while you use the GUI.


In [ ]:
import json as _json
import os
import re
import subprocess
import threading
import time

import requests
import websocket
from tqdm.auto import tqdm

PORT = 5000
CLOUDFLARED_PATH = "/kaggle/working/cloudflared"

if not os.path.exists(CLOUDFLARED_PATH):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CLOUDFLARED_PATH}
    !chmod +x {CLOUDFLARED_PATH}

from colab.webapp import start_server_in_background

start_server_in_background(port=PORT)

# Confirm Flask bound locally before tunneling, so setup failures are obvious.
for _ in range(30):
    try:
        requests.get(f"http://localhost:{PORT}/api/defaults", timeout=2)
        break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    raise RuntimeError(f"Flask server never came up on localhost:{PORT} -- check the install cell above for errors.")
print("local server is up.")

proc = subprocess.Popen(
    [CLOUDFLARED_PATH, "tunnel", "--protocol", "http2", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_found = threading.Event()
tunnel_url = {"value": None}

def _watch_cloudflared():
    for line in proc.stdout:
        print(f"[cloudflared] {line.rstrip()}")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and not url_found.is_set():
            tunnel_url["value"] = match.group(0)
            url_found.set()

threading.Thread(target=_watch_cloudflared, daemon=True).start()
url_found.wait(timeout=30)

if not tunnel_url["value"]:
    raise RuntimeError("cloudflared failed to start a tunnel. Make sure Kaggle Internet is enabled, then re-run this cell.")

print(f"tunnel registered at {tunnel_url['value']}, confirming it is reachable...")
reachable = False
for _ in range(30):
    try:
        r = requests.get(tunnel_url["value"], timeout=5)
        if r.status_code == 200:
            reachable = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)

if reachable:
    print(f"\nopen the GUI here: {tunnel_url['value']}\n")
else:
    print(f"\n{tunnel_url['value']} was registered but did not answer after 30s. Try opening it anyway, or re-run this cell.\n")

API = f"http://localhost:{PORT}/api"
print("watching for conversion jobs started from the GUI (run a job there now)...\n")

while True:
    job_id = None
    while job_id is None:
        job_id = requests.get(f"{API}/jobs/active").json().get("job_id")
        if job_id is None:
            time.sleep(1)

    print(f"=== mirroring job {job_id} ===")
    overall_bar, chapter_bar = None, None
    ws = websocket.create_connection(f"ws://localhost:{PORT}/ws/jobs/{job_id}")
    try:
        while True:
            event = _json.loads(ws.recv())
            kind = event.get("event")

            if kind == "log":
                print(event.get("message", ""))
            elif kind == "plan_ready":
                overall_bar = tqdm(total=event["chunks_total"], unit="chunk", desc="overall", position=0)
            elif kind == "book_start":
                print(f"\n=== book {event['book_index']}/{event['books_total']}: {event['book']} ({event['chapters_total']} chapters) ===")
            elif kind == "chapter_start":
                if chapter_bar is not None:
                    chapter_bar.close()
                chapter_bar = tqdm(total=event["chunks_in_chapter"], unit="chunk", desc=event["chapter_title"][:30], position=1, leave=False)
            elif kind == "chapter_skip":
                print(f"  skip (already synthesized): {event['chapter_title']}")
            elif kind == "chunk_progress":
                if overall_bar is not None:
                    overall_bar.n = event["chunks_done"]
                    eta = event["eta_sec"]
                    overall_bar.set_postfix(rate=f"{event['chunks_per_sec']:.2f}/s", eta=f"{eta:.0f}s" if eta is not None else "?")
                    overall_bar.refresh()
                if chapter_bar is not None:
                    chapter_bar.n = event["chunks_done_chapter"]
                    chapter_bar.refresh()
            elif kind == "chapter_done":
                if chapter_bar is not None:
                    chapter_bar.close()
                    chapter_bar = None
                print(f"  done: {event['chapter_title']}")
            elif kind == "book_done":
                print(f"finished: {event['output_path']}")
            elif kind == "error":
                print(f"ERROR: {event['message']}")
            elif kind == "all_done":
                if overall_bar is not None:
                    overall_bar.close()
                print(f"\nall done: {len(event['outputs'])} audiobook(s) generated")
                for p in event["outputs"]:
                    print(f"  {p}")
            elif kind == "stream_end":
                break
    finally:
        ws.close()

    print("\nwaiting for the next job...\n")
